# Milestone 2 — Inspect FinQA

This notebook inspects the original nested FinQA JSON before any transformation. It uses only **train** and **development** records. The test split is deliberately neither loaded nor inspected, preserving the experiment contract.

Source: the [official FinQA repository](https://github.com/czyssrs/FinQA), pinned to commit `0f16e2867befa6840783e58be38c9efb9229d742`.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from finevid_distill.data.finqa import (
    FINQA_COMMIT,
    INSPECTION_SPLITS,
    candidate_counts,
    gold_format,
    iter_records,
    load_split,
    resolve_gold_evidence,
)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"

# Download only the allowed inspection splits when they are not cached.
splits = {
    split: load_split(RAW_DIR, split, download_if_missing=True)
    for split in INSPECTION_SPLITS
}

print(f"Official FinQA commit: {FINQA_COMMIT}")
print({split: len(records) for split, records in splits.items()})
print("Loaded splits only:", tuple(splits))

Official FinQA commit: 0f16e2867befa6840783e58be38c9efb9229d742
{'train': 6251, 'dev': 883}
Loaded splits only: ('train', 'dev')


## Fields under inspection

| Path | Raw type | Meaning |
|---|---|---|
| `pre_text` | list of strings | Sentences appearing before the associated table. |
| `post_text` | list of strings | Sentences appearing after the associated table. |
| `table` | list of lists of strings | The table as rows and cells. Row 0 often behaves as the header, but some tables have no separate header row. |
| `id` | string | Question/example identifier: source report path plus a question suffix. |
| `qa.question` | string | Natural-language financial question. |
| `qa.gold_inds` | mapping of string to string | Evidence identifier to annotated evidence text. This is resolved exactly below. |
| `qa.program` | string | Gold symbolic reasoning program. |
| `qa.exe_ans` | number or string | Result obtained by executing the gold program. |

In [2]:
sample = splits["train"][0]
schema_rows = [
    ("pre_text", type(sample["pre_text"]).__name__, len(sample["pre_text"])),
    ("post_text", type(sample["post_text"]).__name__, len(sample["post_text"])),
    ("table", type(sample["table"]).__name__, f"{len(sample['table'])} rows"),
    ("id", type(sample["id"]).__name__, sample["id"]),
    ("qa.question", type(sample["qa"]["question"]).__name__, sample["qa"]["question"]),
    ("qa.gold_inds", type(sample["qa"]["gold_inds"]).__name__, sample["qa"]["gold_inds"]),
    ("qa.program", type(sample["qa"]["program"]).__name__, sample["qa"]["program"]),
    ("qa.exe_ans", type(sample["qa"]["exe_ans"]).__name__, sample["qa"]["exe_ans"]),
]
display(pd.DataFrame(schema_rows, columns=["path", "Python type", "example / size"]))

,path,Python type,example / size
0,pre_text,list,15
1,post_text,list,35
2,table,list,4 rows
3,id,str,ADI/2009/page_49.pdf-1
4,qa.question,str,what is the the interest expense in 2009?
5,qa.gold_inds,dict,{'text_1': 'if libor changes by 100 basis poin...
6,qa.program,str,"divide(100, 100), divide(3.8, #0)"
7,qa.exe_ans,float,3.8


## Ten complete raw records

The next cell prints the first ten train objects exactly as loaded. No keys are dropped or flattened; this includes auxiliary fields beyond the eight fields targeted above.

In [3]:
for record_number, record in enumerate(splits["train"][:10], start=1):
    print(f"\n{'=' * 28} RAW RECORD {record_number} {'=' * 28}")
    print(json.dumps(record, indent=2, ensure_ascii=False))


============================ RAW RECORD 1 ============================
{
  "pre_text": [
    "interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .",
    "if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .",
    "foreign currency exposure as more fully described in note 2i .",
    "in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .",
    "dollar-based exposures by entering into forward foreign currency exchange contracts .",
    "the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .",
    "currently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .",
    

## Every observed `gold_inds` format

Two complementary inventories are shown: every key syntax, and every observed combination of evidence sources and supporting-fact count. Together they cover all `gold_inds` shapes in train and development.

In [4]:
format_rows = []
key_rows = []
for split, record in iter_records(splits):
    source_signature, fact_count = gold_format(record)
    format_rows.append(
        {
            "split": split,
            "source_signature": source_signature,
            "supporting_fact_count": fact_count,
            "example_id": record["id"],
        }
    )
    for key in record["qa"]["gold_inds"]:
        if key == "text_-1":
            syntax = "text_-1 (legacy negative-index case)"
        elif re.fullmatch(r"text_\d+", key):
            syntax = "text_<non-negative integer>"
        elif re.fullmatch(r"table_\d+", key):
            syntax = "table_<non-negative integer>"
        else:
            syntax = "UNRECOGNISED"
        key_rows.append({"split": split, "syntax": syntax, "key": key, "id": record["id"]})

format_df = pd.DataFrame(format_rows)
key_df = pd.DataFrame(key_rows)

display(Markdown("### Key syntaxes"))
display(
    key_df.groupby(["syntax", "split"], sort=True)
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda frame: frame.sum(axis=1))
)
display(Markdown("### Exhaustive source-composition × fact-count formats"))
format_summary = (
    format_df.groupby(["source_signature", "supporting_fact_count", "split"], sort=True)
    .agg(records=("example_id", "size"), example_id=("example_id", "first"))
    .reset_index()
)
display(format_summary)
assert "UNRECOGNISED" not in set(key_df["syntax"])

### Key syntaxes

split,dev,train,total
syntax,,,
table_<non-negative integer>,1071,7512,8583
text_-1 (legacy negative-index case),0,1,1
text_<non-negative integer>,442,3179,3621


### Exhaustive source-composition × fact-count formats

,source_signature,supporting_fact_count,split,records,example_id
0,table,1,dev,272,V/2008/page_17.pdf-1
1,table,1,train,1898,AAL/2018/page_13.pdf-2
2,table,2,dev,241,C/2017/page_328.pdf-1
3,table,2,train,1817,INTC/2013/page_71.pdf-4
4,table,3,dev,23,ETR/2003/page_84.pdf-2
5,table,3,train,137,FRT/2009/page_124.pdf-1
6,table,4,dev,4,FRT/2005/page_117.pdf-1
7,table,4,train,44,CME/2010/page_71.pdf-5
8,table,5,dev,1,ETR/2015/page_131.pdf-2
9,table,5,train,16,RE/2006/page_122.pdf-2


## Resolve every gold value to raw evidence

For text, the integer indexes the concatenation `pre_text + post_text`; the annotation value must equal the sentence at that position. For tables, the integer directly indexes `table`, including valid row 0 examples. The value follows FinQA's template: optionally prefix `table[0][0]`, then emit `the ROW_LABEL of COLUMN_HEADER is CELL ;` for each remaining column. Annotation-time spaces before comma, period, or colon are semantically ignored when validating that serialization.

In [5]:
mapping_rows = []
for split, record in iter_records(splits):
    for mapping in resolve_gold_evidence(record):
        mapping_rows.append(
            {
                "split": split,
                "id": record["id"],
                "gold_key": mapping["gold_key"],
                "source_field": mapping["source_field"],
                "source_index": mapping["source_index"],
                "combined_text_index": mapping["combined_text_index"],
                "exact_value_match": mapping["exact_value_match"],
                "normalized_value_match": mapping["normalized_value_match"],
            }
        )

mapping_df = pd.DataFrame(mapping_rows)
print(f"Resolved all {len(mapping_df):,} gold evidence values without an error.")
display(mapping_df.groupby(["source_field", "split"]).size().unstack(fill_value=0))
display(Markdown("### Value-to-source validation"))
display(mapping_df.groupby(["source_field", "exact_value_match"]).size().unstack(fill_value=0))
assert mapping_df["normalized_value_match"].all()
display(Markdown("### The one negative-index annotation"))
display(mapping_df[mapping_df["gold_key"] == "text_-1"])


Resolved all 12,205 gold evidence values without an error.


split,dev,train
source_field,,
post_text,211,1536
pre_text,231,1644
table,1071,7512


### Value-to-source validation

exact_value_match,False,True
source_field,,
post_text,0,1747
pre_text,0,1875
table,305,8278


### The one negative-index annotation

,split,id,gold_key,source_field,source_index,combined_text_index,exact_value_match,normalized_value_match
9597,train,RE/2010/page_120.pdf-2,text_-1,post_text,17,23.0,True,True


## Requested evidence examples

Each example shows the complete fields under inspection followed by the resolved source locations.

In [6]:
def inspected_view(record):
    return {
        "id": record["id"],
        "pre_text": record["pre_text"],
        "post_text": record["post_text"],
        "table": record["table"],
        "qa": {
            "question": record["qa"]["question"],
            "gold_inds": record["qa"]["gold_inds"],
            "program": record["qa"]["program"],
            "exe_ans": record["qa"]["exe_ans"],
        },
    }

def first_matching(predicate):
    return next(record for _, record in iter_records(splits) if predicate(record))

examples = [
    (
        "Prose-only evidence",
        first_matching(lambda record: gold_format(record)[0] == "text"),
    ),
    (
        "Table-only evidence",
        first_matching(lambda record: gold_format(record)[0] == "table"),
    ),
    (
        "Multiple supporting facts (mixed prose and table)",
        first_matching(
            lambda record: gold_format(record)[0] == "table+text"
            and gold_format(record)[1] > 1
        ),
    ),
]

for title, record in examples:
    display(Markdown(f"### {title}: `{record['id']}`"))
    print(json.dumps(inspected_view(record), indent=2, ensure_ascii=False))
    display(pd.DataFrame(resolve_gold_evidence(record)))

### Prose-only evidence: `ADI/2009/page_49.pdf-1`

{
  "id": "ADI/2009/page_49.pdf-1",
  "pre_text": [
    "interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .",
    "if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .",
    "foreign currency exposure as more fully described in note 2i .",
    "in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .",
    "dollar-based exposures by entering into forward foreign currency exchange contracts .",
    "the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .",
    "currently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .",
    "relative to foreign currency exposure

,gold_key,source_kind,annotated_index,source_field,source_index,combined_text_index,annotated_value,raw_evidence,reconstructed_value,exact_value_match,normalized_value_match
0,text_1,text,1,pre_text,1,1,"if libor changes by 100 basis points , our ann...","if libor changes by 100 basis points , our ann...","if libor changes by 100 basis points , our ann...",True,True


### Table-only evidence: `AAL/2018/page_13.pdf-2`

{
  "id": "AAL/2018/page_13.pdf-2",
  "pre_text": [
    "the following table shows annual aircraft fuel consumption and costs , including taxes , for our mainline and regional operations for 2018 , 2017 and 2016 ( gallons and aircraft fuel expense in millions ) .",
    "year gallons average price per gallon aircraft fuel expense percent of total operating expenses ."
  ],
  "post_text": [
    "as of december 31 , 2018 , we did not have any fuel hedging contracts outstanding to hedge our fuel consumption .",
    "as such , and assuming we do not enter into any future transactions to hedge our fuel consumption , we will continue to be fully exposed to fluctuations in fuel prices .",
    "our current policy is not to enter into transactions to hedge our fuel consumption , although we review that policy from time to time based on market conditions and other factors .",
    "fuel prices have fluctuated substantially over the past several years .",
    "we cannot predict the future availabil

,gold_key,source_kind,annotated_index,source_field,source_index,combined_text_index,annotated_value,raw_evidence,reconstructed_value,exact_value_match,normalized_value_match
0,table_1,table,1,table,1,None,year the 2018 of gallons is 4447 ; the 2018 of...,"[2018, 4447, $ 2.23, $ 9896, 23.6% ( 23.6 % )]",year the 2018 of gallons is 4447 ; the 2018 of...,True,True


### Multiple supporting facts (mixed prose and table): `ABMD/2012/page_75.pdf-1`

{
  "id": "ABMD/2012/page_75.pdf-1",
  "pre_text": [
    "abiomed , inc .",
    "and subsidiaries notes to consolidated financial statements 2014 ( continued ) note 8 .",
    "stock award plans and stock-based compensation ( continued ) restricted stock and restricted stock units the following table summarizes restricted stock and restricted stock unit activity for the fiscal year ended march 31 , 2012 : number of shares ( in thousands ) weighted average grant date fair value ( per share ) ."
  ],
  "post_text": [
    "the remaining unrecognized compensation expense for outstanding restricted stock and restricted stock units , including performance-based awards , as of march 31 , 2012 was $ 7.1 million and the weighted-average period over which this cost will be recognized is 2.2 years .",
    "the weighted average grant-date fair value for restricted stock and restricted stock units granted during the years ended march 31 , 2012 , 2011 , and 2010 was $ 18.13 , $ 10.00 and $ 7.67 per s

,gold_key,source_kind,annotated_index,source_field,source_index,combined_text_index,annotated_value,raw_evidence,reconstructed_value,exact_value_match,normalized_value_match
0,table_2,table,2,table,2,NaN,the granted of number of shares ( in thousands...,"[granted, 607, 18.13]",the granted of number of shares ( in thousands...,True,True
1,text_15,text,15,post_text,12,15.0,"during the year ended march 31 , 2012 , the co...","during the year ended march 31 , 2012 , the co...","during the year ended march 31 , 2012 , the co...",True,True


## Candidate-count distributions

At this raw-schema stage, an evidence candidate is one `pre_text` sentence, one `post_text` sentence, or one raw table row. Every table row is counted because `table_0` can itself be gold evidence. These are evidence-unit counts inside the associated positive report—not the approximately eight-report training sets that a later transformation milestone will build.

In [7]:
candidate_rows = []
for split, record in iter_records(splits):
    candidate_rows.append({"split": split, "id": record["id"], **candidate_counts(record)})
candidate_df = pd.DataFrame(candidate_rows)
count_columns = [
    "pre_text_candidates",
    "post_text_candidates",
    "prose_candidates",
    "table_row_candidates",
    "all_evidence_candidates",
    "gold_supporting_facts",
]

display(Markdown("### Quantile summary by split"))
distribution_summary = (
    candidate_df.groupby("split")[count_columns]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    .round(2)
)
display(distribution_summary)

display(Markdown("### Total evidence-candidate buckets"))
candidate_df["candidate_bucket"] = pd.cut(
    candidate_df["all_evidence_candidates"],
    bins=[0, 10, 20, 30, 40, 50, 100, float("inf")],
    labels=["1–10", "11–20", "21–30", "31–40", "41–50", "51–100", "101+"],
)
display(candidate_df.groupby(["candidate_bucket", "split"], observed=False).size().unstack(fill_value=0))

display(Markdown("### Exact gold-support count distribution"))
display(
    candidate_df.groupby(["gold_supporting_facts", "split"])
    .size()
    .unstack(fill_value=0)
)

### Quantile summary by split

pre_text_candidates                                                 \
                    count   mean    std  min  25%  50%   75%   90%   95%   
split                                                                      
dev                 883.0  11.02  10.10  1.0  3.0  8.0  17.0  24.0  28.9   
train              6251.0  11.59  13.31  1.0  3.0  9.0  17.0  23.0  27.0   

              ... gold_supporting_facts                                      \
         99%  ...                  mean   std  min  25%  50%  75%  90%  95%   
split         ...                                                             
dev    38.26  ...                  1.71  0.88  1.0  1.0  2.0  2.0  3.0  3.0   
train  41.00  ...                  1.71  0.85  1.0  1.0  2.0  2.0  3.0  3.0   

                  
        99%  max  
split             
dev    4.18  8.0  
train  5.00  9.0  

[2 rows x 66 columns]

### Total evidence-candidate buckets

split,dev,train
candidate_bucket,,
1–10,20,180
11–20,158,821
21–30,271,2570
31–40,323,1948
41–50,87,493
51–100,19,192
101+,5,47


### Exact gold-support count distribution

split,dev,train
gold_supporting_facts,,
1,416,2850
2,356,2720
3,80,453
4,22,155
5,2,39
6,3,22
7,3,6
8,1,0
9,0,6


## Schema conclusion: exact `gold_inds` mapping

A FinQA record binds one question (`qa.question`), one gold program (`qa.program`), and its executed answer (`qa.exe_ans`) to a source report identified by `id`. The report context is split into ordered prose lists (`pre_text`, then `post_text`) and a two-dimensional string array (`table`).

`qa.gold_inds` is a dictionary of **evidence identifier → annotated evidence text**:

1. For `text_N`, parse `N` as an integer and index `combined_text = pre_text + post_text`. If the resolved position is smaller than `len(pre_text)`, it maps to `pre_text[N]`; otherwise it maps to `post_text[N - len(pre_text)]`. The dictionary value is exactly the raw sentence at that position. Duplicate sentence strings are therefore disambiguated by the index, not by searching text. There is one train annotation, `text_-1`, whose value is exactly the final combined sentence; it must be preserved as a documented legacy negative-index case and normalized to its non-negative position before downstream candidate IDs are created.
2. For `table_N`, `N` is the zero-based raw row index and maps directly to `table[N]`. Row 0 is valid gold evidence in some records, so it must not be dropped unconditionally as a header. The value is reconstructed by prefixing `table[0][0]` when non-empty, then, for every paired `table[0][1:]` header and `table[N][1:]` cell, appending `the {table[N][0]} of {header} is {cell} ;`. In the pinned train/dev files, 305 table values differ from this reconstruction only by annotation-time spaces before comma, period, or colon; punctuation-spacing normalization makes all 8,583 table values match. The source evidence remains the raw cell list `table[N]`.

Across the inspected train and development splits, every key matches one of those forms and every annotated value resolves under these rules. The test split was not loaded.

In [8]:
expected_gold_values = sum(
    len(record["qa"]["gold_inds"])
    for _, record in iter_records(splits)
)
assert len(mapping_df) == expected_gold_values
assert set(mapping_df["source_field"]) == {"pre_text", "post_text", "table"}
assert len(mapping_df[mapping_df["gold_key"] == "text_-1"]) == 1
assert mapping_df["normalized_value_match"].all()
assert set(splits) == {"train", "dev"}
print(
    f"Schema mapping complete: {len(mapping_df):,} / {expected_gold_values:,} "
    "train+dev gold values resolved; test untouched."
)

Schema mapping complete: 12,205 / 12,205 train+dev gold values resolved; test untouched.
